<a href="https://colab.research.google.com/github/zorGizem/Erken-Evre-Alzhemir-Tespiti/blob/main/notebooks/registrasion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Uzaysal Normalizasyon (MNI152 Registration)
Farklı bireylerden alınan MR görüntüleri; kafa yapısı, boyutu ve beyin oryantasyonu açısından doğal bir çeşitlilik gösterir. Bu çeşitlilik, makine öğrenimi modellerinin doku değişimlerini standart bir düzlemde analiz etmesini zorlaştırır. Uzaysal Normalizasyon (Registration) işlemi, tüm beyin görüntülerini dünya çapında standart kabul edilen MNI152 (Montreal Neurological Institute) şablonu üzerine hizalayarak verileri ortak bir koordinat sistemine taşır.

# Neden MNI Hizalaması Tercih Edildi?
Bireyler arasındaki kafa yapısı ve beyin oryantasyonu farklılıklarını gidermek amacıyla, tüm görüntüler dünya genelinde standart kabul edilen MNI152 koordinat sistemine taşınmıştır. Bu süreçte kullanılan ANTS SyN algoritması, beyin dokusundaki şekil bozukluklarını doğrusal olmayan bir şekilde düzelterek hipokampus gibi kritik bölgelerin tüm hastalarda aynı koordinatlara gelmesini sağlar. Skull stripping aşamasından gelen maskelerin sürece dahil edilmesiyle algoritmanın doğrudan beyin dokusuna odaklanması sağlanmış, böylece derin öğrenme modelinin Alzheimer kaynaklı doku kayıplarını en yüksek hassasiyetle ve standart bir veri yapısı üzerinden öğrenmesi garanti altına alınmıştır.

In [ ]:
!pip install antspyx -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 61.3 MB/s eta 0:00:00


In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU Aktif: {torch.cuda.get_device_name(0)}")
else:
    print("DİKKAT: GPU bulunamadı!")

GPU Aktif: NVIDIA A100-SXM4-40GB


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
CONFIG_CN = {
    "kategori"    : "CN",
    "kaynak": '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/skull_stripingg/skull_striping_CN',
    "hedef" : '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/Registrasionnnn/Registrasion_CN'
}


In [ ]:
CONFIG_EMCI = {
    "kategori"    : "EMCI",
    "kaynak": '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/skull_stripingg/skull_striping_EMCI',
    "hedef" : '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/Registrasionnnn/Registrasion_EMCI'
}


In [ ]:
import ants

mni_sablon = ants.get_ants_data('mni')
mni_img    = ants.image_read(mni_sablon)

print(f"MNI Şablonu : {mni_sablon}")
print(f"Boyut       : {mni_img.shape}")
print(f"Spacing     : {mni_img.spacing}")

MNI Şablonu : /root/.antspy/mni.nii.gz
Boyut       : (182, 218, 182)
Spacing     : (1.0, 1.0, 1.0)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import numpy as np
import ants

def canli_donusum_simulasyonu(moving_img, warped_img, fixed_img):
    """
    Orijinal beyin ile MNI'a hizalanmış beyin arasındaki dönüşümü
    adım adım gösteren bir animasyon oluşturur.
    """
    # Görüntüyü yeniden boyutlandır (sadece çözünürlük eşitlemesi)
    moving_resampled = ants.resample_image_to_target(moving_img, fixed_img)

    # Tam orta noktadan yatay bir kesit (axial) al
    slice_idx = fixed_img.shape[2] // 2
    ilk_hal = moving_resampled.numpy()[:, :, slice_idx]
    son_hal = warped_img.numpy()[:, :, slice_idx]

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.axis('off')
    ax.set_title("MNI Şablonuna Dönüşüm (Morphing)...")

    im = ax.imshow(ilk_hal, cmap='gray')
    kare_sayisi = 40

    def update(frame):
        alpha = frame / (kare_sayisi - 1)
        anlik_goruntu = (1.0 - alpha) * ilk_hal + alpha * son_hal
        im.set_array(anlik_goruntu)
        return [im]

    ani = FuncAnimation(fig, update, frames=kare_sayisi, interval=100, blit=True)
    plt.close()

    return HTML(ani.to_jshtml())

In [ ]:
import os
import ipywidgets as widgets
from IPython.display import display, clear_output

# Yukarıda tanımladığın CONFIG_CN'in kaynak yolunu otomatik alıyoruz
kaynak_klasor = CONFIG_CN["kaynak"]

print("Klasör taranıyor, hiyerarşi oluşturuluyor...")

# Veri yapısı: { "Hasta_ID": [ ("Seans_Adi", "Tam_Yol"), ... ] }
hasta_seans_sozlugu = {}

# Klasörleri Hasta -> Seans hiyerarşisine göre tarıyoruz
for subject_id in sorted(os.listdir(kaynak_klasor)):
    subject_path = os.path.join(kaynak_klasor, subject_id)
    if os.path.isdir(subject_path):
        seanslar_listesi = []
        for seans in sorted(os.listdir(subject_path)):
            seans_yolu = os.path.join(subject_path, seans)
            if os.path.isdir(seans_yolu):
                # Seans içindeki .nii.gz dosyasını bul (maske olmayan)
                for file in os.listdir(seans_yolu):
                    if file.endswith('.nii.gz') and '_bet' not in file:
                        tam_yol = os.path.join(seans_yolu, file)
                        seanslar_listesi.append((seans, tam_yol))

        # Eğer hastanın geçerli dosyası varsa sözlüğe ekle
        if seanslar_listesi:
            hasta_seans_sozlugu[subject_id] = seanslar_listesi

# 1. Menü: Hasta Seçimi
hasta_dropdown = widgets.Dropdown(
    options=list(hasta_seans_sozlugu.keys()),
    description='1. Hasta Seç:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='250px')
)

# 2. Menü: Seans Seçimi (İlk başta 1. menüdeki ilk hastanın seansları yüklenir)
baslangic_hastasi = hasta_dropdown.value
seans_dropdown = widgets.Dropdown(
    options=hasta_seans_sozlugu[baslangic_hastasi] if baslangic_hastasi else [],
    description='2. Seans Seç:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

buton = widgets.Button(
    description='Dönüşümü Canlandır',
    button_style='info',
    icon='play'
)

cikis_ekrani = widgets.Output()

# --- Hasta değiştiğinde Seans menüsünü güncelleyen fonksiyon ---
def hasta_degisti(change):
    if change['type'] == 'change' and change['name'] == 'value':
        yeni_hasta = change['new']
        # 2. menünün seçeneklerini yeni hastanın seanslarıyla değiştir
        seans_dropdown.options = hasta_seans_sozlugu[yeni_hasta]

hasta_dropdown.observe(hasta_degisti)
# -----------------------------------------------------------------

def animasyonu_baslat(b):
    with cikis_ekrani:
        clear_output(wait=True)
        secilen_dosya_yolu = seans_dropdown.value

        if not secilen_dosya_yolu:
            print("Lütfen geçerli bir dosya seçin!")
            return

        print(f"İşleniyor: Hasta {hasta_dropdown.value} -> Seans {seans_dropdown.label}")
        print("ANTs Registration çalışıyor (Bu işlem 1-2 dakika sürebilir)...")

        # Görüntüyü oku ve hizala
        moving_img = ants.image_read(secilen_dosya_yolu)
        reg = ants.registration(fixed=mni_img, moving=moving_img, type_of_transform='antsRegistrationSyNQuick[a]')

        print("Kayıtlama tamamlandı! Animasyon oynatılıyor...")

        # Animasyonu oluştur ve göster
        animasyon_html = canli_donusum_simulasyonu(moving_img, reg['warpedmovout'], mni_img)
        display(animasyon_html)

buton.on_click(animasyonu_baslat)

clear_output()
# Arayüzü yan yana güzel bir şekilde diziyoruz
display(widgets.HBox([hasta_dropdown, seans_dropdown, buton]), cikis_ekrani)

In [ ]:
import os
import shutil
import ants

def batch_mni_hizalama(config, mni_sablon, batch_size=20):
    source_base = config["kaynak"]
    output_base = config["hedef"]
    etiket = config["kategori"]

    local_in = "/content/batch_in_mni"
    local_out = "/content/batch_out_mni"

    print(f"\n {etiket} MNI Registration Başlıyor (Grup Boyutu: {batch_size})")

    # 1. Analiz Aşaması
    all_pending = []
    subjects = sorted([s for s in os.listdir(source_base) if os.path.isdir(os.path.join(source_base, s))])

    toplam_dosya = 0
    atlanan_dosya = 0
    basarili_dosya = 0
    hatali_dosya = 0

    print(" Klasörler taranıyor, lütfen bekleyin...", end="\r")

    for subject_id in subjects:
        subject_path = os.path.join(source_base, subject_id)
        seanslar = [d for d in os.listdir(subject_path) if os.path.isdir(os.path.join(subject_path, d))]

        for seans in seanslar:
            seans_yolu = os.path.join(subject_path, seans)
            dosyalar = os.listdir(seans_yolu)
            beyin_dosyasi = next((f for f in dosyalar if f.endswith('.nii.gz') and '_bet' not in f), None)
            maske_dosyasi = next((f for f in dosyalar if f.endswith('_bet.nii.gz')), None)

            if beyin_dosyasi:
                toplam_dosya += 1
                rel_path = os.path.join(subject_id, seans)
                cikti_yolu = os.path.join(output_base, rel_path, beyin_dosyasi)

                if not os.path.exists(cikti_yolu):
                    all_pending.append((os.path.join(seans_yolu, beyin_dosyasi),
                                      os.path.join(seans_yolu, maske_dosyasi) if maske_dosyasi else None,
                                      rel_path, beyin_dosyasi, subject_id))
                else:
                    atlanan_dosya += 1

    # İşleme başlamadan önce özet yapısı veriliyor.
    kalan_dosya = len(all_pending)
    print(" "*50, end="\r")
    print(f" KLASÖR ANALİZİ ({etiket}):")
    print(f"    Toplam Bulunan Dosya : {toplam_dosya}")
    print(f"    Zaten İşlenmiş Olan  : {atlanan_dosya}")
    print(f"    Şimdi İşlenecek Olan : {kalan_dosya}")
    print("-" * 40)


    if not all_pending:
        print(f" İşlenecek yeni dosya yok. Sistem durduruluyor.")
    else:

        for i in range(0, kalan_dosya, batch_size):
            batch = all_pending[i:i+batch_size]
            shutil.rmtree(local_in, ignore_errors=True)
            shutil.rmtree(local_out, ignore_errors=True)
            os.makedirs(local_in, exist_ok=True)
            os.makedirs(local_out, exist_ok=True)

            batch_subjects = sorted(list(set([item[4] for item in batch])))
            print(f"\n Grup {i//batch_size + 1} Başlıyor...")
            print(f" İşlenen Hastalar: {', '.join(batch_subjects)}")

            mapping = {}
            for full_src, mask_src, rel_p, fname, sid in batch:
                unique_name = rel_p.replace("/", "_") + "___" + fname
                local_girdi = os.path.join(local_in, unique_name)
                shutil.copy2(full_src, local_girdi)

                local_maske = None
                if mask_src:
                    local_maske = os.path.join(local_in, unique_name.replace(".nii.gz", "_mask.nii.gz"))
                    shutil.copy2(mask_src, local_maske)

                mapping[unique_name] = (rel_p, fname, local_girdi, local_maske)

            print(f" MNI Hizalama  çalışıyor... ", end="", flush=True)

            try:
                for u_name, (rel_p, fname, l_girdi, l_maske) in mapping.items():
                    l_out_file = os.path.join(local_out, u_name)
                    img = ants.image_read(l_girdi)
                    if l_maske:
                        mask = ants.image_read(l_maske)
                        img = img * mask
                        reg = ants.registration(fixed=mni_sablon, moving=img, type_of_transform='antsRegistrationSyNQuick[s]', moving_mask=mask)
                    else:
                        reg = ants.registration(fixed=mni_sablon, moving=img, type_of_transform='antsRegistrationSyNQuick[s]')
                    ants.image_write(reg['warpedmovout'], l_out_file)

                print(" Bitti")

                print(f" Drive'a aktarılıyor... ", end="", flush=True)
                for u_name, (rel_p, fname, _, _) in mapping.items():
                    l_res = os.path.join(local_out, u_name)
                    d_target_dir = os.path.join(output_base, rel_p)
                    os.makedirs(d_target_dir, exist_ok=True)
                    if os.path.exists(l_res):
                        shutil.move(l_res, os.path.join(d_target_dir, fname))
                        basarili_dosya += 1


                su_an_biten = min(i+batch_size, kalan_dosya)
                print(f" {su_an_biten}/{kalan_dosya} yeni dosya kaydedildi.")

            except Exception as e:
                print(f" Grup Hatası: {str(e)}")
                hatali_dosya += len(batch)

    # Kodun sonunda özet bilgisi veriliyor.
    print(f"\n{etiket} MNI Registration Özeti:")
    print(f"Toplam   : {toplam_dosya}")
    print(f"Başarılı : {basarili_dosya}")
    print(f"Atlanan  : {atlanan_dosya}")
    print(f"Hatalı   : {hatali_dosya}")
    print(f"{etiket} grubu tamamlandı.")

In [ ]:
batch_mni_hizalama(CONFIG_CN, mni_img)

In [ ]:
batch_mni_hizalama(CONFIG_EMCI, mni_img)


 EMCI MNI Registration Başlıyor (Grup Boyutu: 20)
 KLASÖR ANALİZİ (EMCI):
    Toplam Bulunan Dosya : 211
    Zaten İşlenmiş Olan  : 160
    Şimdi İşlenecek Olan : 51
----------------------------------------

 Grup 1 Başlıyor...
 İşlenen Hastalar: 053_S_2357, 053_S_2396, 053_S_4557, 053_S_4661, 057_S_2398, 067_S_2195, 067_S_2196, 067_S_4072, 067_S_4310, 067_S_5160, 068_S_2168
 MNI Hizalama  çalışıyor...  Bitti
 Drive'a aktarılıyor...  20/51 yeni dosya kaydedildi.

 Grup 2 Başlıyor...
 İşlenen Hastalar: 068_S_2168, 068_S_2171, 068_S_2187, 068_S_2193, 068_S_2248, 068_S_2315, 068_S_2316, 068_S_4067, 068_S_4217, 068_S_4332, 068_S_4431
 MNI Hizalama  çalışıyor...  Bitti
 Drive'a aktarılıyor...  40/51 yeni dosya kaydedildi.

 Grup 3 Başlıyor...
 İşlenen Hastalar: 068_S_4914, 070_S_4708, 072_S_2026, 072_S_2027, 072_S_2037, 072_S_2072
 MNI Hizalama  çalışıyor...  Bitti
 Drive'a aktarılıyor...  51/51 yeni dosya kaydedildi.

EMCI MNI Registration Özeti:
Toplam   : 211
Başarılı : 51
Atlanan  : 16

In [ ]:
import os

def mni_kontrol(config):
    hedef  = config["hedef"]
    etiket = config["kategori"]

    goruntuler_sayisi = 0
    for root, dirs, files in os.walk(hedef):
        for file in files:
            if file.endswith('.nii.gz'):
                goruntuler_sayisi += 1

    print(f"{etiket} MNI Klasörü:")
    print(f"   Hizalanmış görüntü sayısı : {goruntuler_sayisi}")
    print(f"    Tüm işlenmiş görüntülerin kontrolü tamamlandı.")

mni_kontrol(CONFIG_CN)
print()
mni_kontrol(CONFIG_EMCI)

CN MNI Klasörü:
   Hizalanmış görüntü sayısı : 266
    Tüm işlenmiş görüntülerin kontrolü tamamlandı.

EMCI MNI Klasörü:
   Hizalanmış görüntü sayısı : 211
    Tüm işlenmiş görüntülerin kontrolü tamamlandı.
